# Across-barcode taxonomic and PCR-plate prevalence summaries
This replaces the old consensus-derived target-species display with direct
SINTAX read evidence. The tables retain source_folder and PCR_plate.
Red flags represent occurrence in >=80% of analysed PCR-plate codes at the
same taxonomic rank. Broad ranks can naturally be widespread.


## Primer naming
The sequence-based dictionary maps original MFT names (keys), such as
`forward_tagged_primer_1` and `reverse_tagged_primer_1`, to PCR names (values),
`bc1002` and `bc1100rc`, respectively. These numbers are not interchangeable.
Only the dictionary values are exported in biological tables: `forward_PCR_primer`
and `reverse_PCR_primer` at read/primer-pair level, and `forward_primers_detected`
and `reverse_primers_detected` in aggregated summaries. Lists contain unique,
semicolon-separated values. The mapping is saved separately as JSON for provenance.


In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'code/sintax_analysis.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'code'))
ANALYSIS_DIR = Path(os.environ.get('SINTAX_ANALYSIS_ROOT', str(PROJECT_ROOT / 'data/sintax_analysis')))

import matplotlib.pyplot as plt
species = pd.read_csv(ANALYSIS_DIR / 'all_species_summary.tsv', sep='	')
prevalence = pd.read_csv(ANALYSIS_DIR / 'taxon_plate_prevalence.tsv', sep='	')
qc = pd.read_csv(ANALYSIS_DIR / 'barcode_analysis_summary.tsv', sep='	')
display(qc)
display(prevalence[prevalence.contamination.eq('contamination')])


In [ ]:
from sintax_analysis import join_values
# Species-by-folder table: no aggregation across distinct plate/sample identities.
counts = species.pivot_table(index=['source_folder','PCR_plate'], columns='species', values='read_count', aggfunc='sum', fill_value=0)
counts = counts.join(qc.set_index(['source_folder','PCR_plate'])[['forward_primers_detected','reverse_primers_detected']])
counts.to_csv(ANALYSIS_DIR / 'species_by_folder.tsv', sep='	', index=True, header=True, mode='x')
flag_counts = (species[species.read_count.gt(0)].groupby(['source_folder','PCR_plate','contamination'])
    .agg(species_count=('species','nunique'), read_count=('read_count','sum'),
         forward_primers_detected=('forward_primers_detected',join_values),
         reverse_primers_detected=('reverse_primers_detected',join_values)).reset_index())
flag_counts.to_csv(ANALYSIS_DIR / 'contamination_summary_by_folder.tsv', sep='	', index=False, header=True, mode='x')
display(flag_counts)


In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
qc.plot.bar(x='source_folder', y=['assigned_query_count','species_query_count'], ax=ax)
ax.set(ylabel='Cleaned query count', xlabel='Source folder', title='SINTAX confident assignments by folder')
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / 'barcode_assignment_counts.png', dpi=160)
plt.show()
